# Citi Bike Station Activity and Imbalance — June 2026
**Matthew Wiecha** · Urban planning and transportation data analysis

This analysis explores how trip arrivals and departures vary across New York City Citi Bike stations during weekday morning commuting hours. It compares **net flow** and **total activity** to identify stations with relatively strong directional pressure.

**Interpretation:** These are *indicators of potential bike or dock pressure*, **not measured stockouts or verified operational failures**. Actual availability depends on station capacity, initial inventory, and rebalancing.

## 1. Data and setup

Input: six June 2026 Citi Bike trip-data CSV files, named `202606-citibike-tripdata_1.csv` through `_6.csv`, stored in the local `data/` directory. Obtain trip records from the [Citi Bike System Data](https://citibikenyc.com/system-data) page. The raw datasets are not included here.

This notebook requires `pandas`, `matplotlib`, `numpy`, `openpyxl`, and `jupyter`. Outputs are intentionally cleared; rerun with source data to generate tables and charts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

files = sorted(Path('data').glob('202606-citibike-tripdata_*.csv'))
if not files:
    raise FileNotFoundError('Add the June 2026 Citi Bike CSV files to a data/ folder.')
print(f'Found {len(files)} CSV files')
citibike = pd.concat((pd.read_csv(path, low_memory=False) for path in files), ignore_index=True)
print(f'{len(citibike):,} trip records and {len(citibike.columns)} columns')
display(citibike.head())

## 2. Inspect and prepare trips

Parse timestamps, inspect missing values, and derive simple temporal variables. The dataset contains trip-level records, not a station-inventory time series.

In [ ]:
citibike['started_at'] = pd.to_datetime(citibike['started_at'], errors='coerce')
citibike['ended_at'] = pd.to_datetime(citibike['ended_at'], errors='coerce')
print('Missing values by column:')
display(citibike.isna().sum().sort_values(ascending=False))

citibike['trip_minutes'] = (citibike['ended_at'] - citibike['started_at']).dt.total_seconds() / 60
citibike['hour'] = citibike['started_at'].dt.hour
citibike['is_weekend'] = citibike['started_at'].dt.dayofweek >= 5
print('Rider type:')
display(citibike['member_casual'].value_counts(dropna=False))
print('Bike type:')
display(citibike['rideable_type'].value_counts(dropna=False))

## 3. Overall daily patterns

Trip starts by hour, comparing weekdays and weekends. These are **monthly totals by hour**, not average trips per individual day; the number of weekdays and weekend days differs.

In [ ]:
hourly = citibike.groupby('hour').size().reindex(range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hourly.index, hourly.values)
ax.set(xticks=range(24), xlabel='Start hour', ylabel='June trip starts', title='Citi Bike trip starts by hour — June 2026')
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

by_day_type = (citibike.groupby(['is_weekend','hour']).size().unstack(0, fill_value=0).reindex(range(24), fill_value=0))
fig, ax = plt.subplots(figsize=(10, 5))
for is_weekend, label in [(False,'Weekday'),(True,'Weekend')]:
    if is_weekend in by_day_type.columns:
        ax.plot(by_day_type.index, by_day_type[is_weekend], label=label)
ax.set(xticks=range(24), xlabel='Start hour', ylabel='June trip starts', title='Weekday and weekend trip-start totals')
ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 4. Focus on the weekday morning commute

Select **weekday trips starting between 7:00 a.m. and 9:59 a.m.** This reproduces the selection rule used in the original analysis.

**Important scope limitation:** The *arrival events* below use trip end times, which may be after 10:00 a.m. Because the subset is selected by **trip start**, it is not a strict 7–10 a.m. inventory-change series. Interpret the labels as morning-start trip cohort events, not all station arrivals/departures in a bounded time window.

In [ ]:
morning_peak = citibike.loc[(~citibike['is_weekend']) & citibike['hour'].between(7, 9)].copy()
print(f'{len(morning_peak):,} weekday morning-start trips')

weekday_riders = (morning_peak.groupby(['hour','member_casual']).size().unstack(fill_value=0))
fig, ax = plt.subplots(figsize=(9,5))
for rider_type in weekday_riders.columns:
    ax.plot(weekday_riders.index, weekday_riders[rider_type], marker='o', label=rider_type)
ax.set(xlabel='Start hour', ylabel='Trip starts in June', title='Morning-start trips by rider type')
ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 5. Construct station arrival and departure events

Each trip contributes a departure event at its start station/time and an arrival event at its end station/time. Group events by station and 15-minute clock intervals. Records with missing station names/timestamps are excluded from station-level calculations.

In [ ]:
departures = morning_peak[['started_at','start_station_name']].rename(columns={'started_at':'time','start_station_name':'station'})
departures['event'] = 'departure'
arrivals = morning_peak[['ended_at','end_station_name']].rename(columns={'ended_at':'time','end_station_name':'station'})
arrivals['event'] = 'arrival'
events = pd.concat([departures, arrivals], ignore_index=True).dropna(subset=['station','time'])
events['interval'] = events['time'].dt.floor('15min')

station_activity = (events.groupby(['station','interval','event']).size().unstack(fill_value=0).reset_index())
for event in ['arrival','departure']:
    if event not in station_activity:
        station_activity[event] = 0
station_activity['net_flow'] = station_activity['arrival'] - station_activity['departure']
station_activity['total_activity'] = station_activity['arrival'] + station_activity['departure']
station_activity['date'] = station_activity['interval'].dt.date
station_activity['time'] = station_activity['interval'].dt.time
display(station_activity.head())

## 6. Compare station-level imbalances across weekdays

Calculate the mean for each **station and clock-time combination** across June dates **with observed events in that 15-minute bin**. Select the lowest average net flow (potential bike pressure) and highest (potential dock pressure) for each station.

**Statistical limitation:** Zero-activity days are absent from the grouping, so averages are conditional on observed activity rather than calendar-day averages. Missing inventory data and station-capacity variation prevent direct conclusions about real stockouts.

In [ ]:
average_activity = (station_activity.groupby(['station','time'], as_index=False)
                    .agg(arrival=('arrival','mean'), departure=('departure','mean'),
                         net_flow=('net_flow','mean'), total_activity=('total_activity','mean')))
bike_stress = (average_activity.sort_values('net_flow').groupby('station', as_index=False).first())
dock_stress = (average_activity.sort_values('net_flow', ascending=False).groupby('station', as_index=False).first())
print('Strongest average negative net flow:')
display(bike_stress.nsmallest(10,'net_flow')[['station','time','net_flow','total_activity']])
print('Strongest average positive net flow:')
display(dock_stress.nlargest(10,'net_flow')[['station','time','net_flow','total_activity']])

## 7. Visualize net flow and activity together

Each station is represented by whichever extreme average net flow has the larger absolute magnitude, matching the original analytical approach. The **x-axis** represents signed average net flow (negative = more departures; positive = more arrivals) and the **y-axis** represents average event activity in the selected bin.

Because net flow is a difference and total activity is a sum of nonnegative counts, `abs(net_flow) <= total_activity`, which contributes to the V-shaped geometry of the scatterplot. Large activity alone does not establish station pressure.

In [ ]:
bike_master = bike_stress[['station','time','net_flow','total_activity']].copy()
bike_master['stress_type'] = 'Bike pressure'
dock_master = dock_stress[['station','time','net_flow','total_activity']].copy()
dock_master['stress_type'] = 'Dock pressure'
master = pd.concat([bike_master,dock_master], ignore_index=True)
master['abs_net_flow'] = master['net_flow'].abs()
master = master.sort_values('abs_net_flow', ascending=False).drop_duplicates('station')

fig, ax = plt.subplots(figsize=(10,7))
ax.scatter(master['net_flow'], master['total_activity'], alpha=.6)
ax.axvline(0, linewidth=1, linestyle='--')
for condition in [master['net_flow'] < 0, master['net_flow'] > 0]:
    subset = master.loc[condition]
    if not subset.empty:
        row = subset.sort_values('total_activity', ascending=False).iloc[0]
        ax.annotate(row['station'], xy=(row['net_flow'],row['total_activity']),
                    xytext=(7,7), textcoords='offset points', fontsize=9,
                    arrowprops={'arrowstyle':'-', 'linewidth':.6})
ax.set(xlabel='Average net flow in selected 15-minute bin',
       ylabel='Average total activity in same bin',
       title='Citi Bike station imbalance and activity — weekday morning trip cohort')
plt.tight_layout()
plt.savefig('station_imbalance_scatter.png', dpi=180, bbox_inches='tight')
plt.show()

display(master[['station','time','net_flow','total_activity','stress_type']].head(20))

## 8. Export station-level results

The exported table contains the selected time window, average net flow, average activity, and the direction of pressure for each station. The image and spreadsheet are generated when the notebook is run.

In [ ]:
master.to_csv('citibike_station_imbalance_summary.csv', index=False)
print(f'Exported {len(master):,} station summaries')

## Interpretation and next steps

These results identify places worth investigating further, not proven bike/dock shortages. A more complete operational analysis would integrate **historical station inventories, station capacity, and rebalancing activity**, and would include zero-event dates explicitly when computing average clock-time flows. A strict inventory-change analysis should also define station events by **event timestamp** within the target window rather than trip start time alone.

*Source: Citi Bike trip history data, June 2026. Analysis by Matthew Wiecha.*